In [0]:
# Databricks notebook source
import pyspark.sql.functions as F

storage_account = "vesseldatavd99"
storage_key = "TUqbHBJ5+ZvfOHAXTGwH75HDC9QAcAqMXinMWhfAa3Xo9rSwnkQyOzRBAAPVtToDpGnT+ZudKdII+AStmezehA=="   # placeholder: paste the key only in your running notebook, never commit it

spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
RAW = f"abfss://raw@{storage_account}.dfs.core.windows.net/AIS_2019_01_*.csv"

# COMMAND ----------
# All strings: nothing gets silently nulled by a schema
raw = (spark.read.option("header", True).option("inferSchema", False).csv(RAW)
       .withColumn("file", F.col("_metadata.file_name")))
raw.printSchema()
display(raw.limit(10))

total = raw.count()
print(f"Total rows: {total:,}")
display(raw.groupBy("file").count().orderBy("file"))

# COMMAND ----------
# Null/empty rate per column
display(raw.select([
    (F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)) / total * 100).alias(c)
    for c in raw.columns if c != "file"]))

root
 |-- MMSI: string (nullable = true)
 |-- BaseDateTime: string (nullable = true)
 |-- LAT: string (nullable = true)
 |-- LON: string (nullable = true)
 |-- SOG: string (nullable = true)
 |-- COG: string (nullable = true)
 |-- Heading: string (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Length: string (nullable = true)
 |-- Width: string (nullable = true)
 |-- Draft: string (nullable = true)
 |-- Cargo: string (nullable = true)
 |-- TransceiverClass: string (nullable = true)
 |-- file: string (nullable = false)

Total rows: 49,208,485


In [0]:
# code 0
# Databricks notebook source
import pyspark.sql.functions as F

# Paste the NEW (rotated) key into the widget box at the top of the notebook.
# It is not stored in the code, so this file is safe to commit.


storage_account = "vesseldatavd99"
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
RAW = f"abfss://raw@{storage_account}.dfs.core.windows.net/AIS_2019_01_*.csv"

# COMMAND ----------
# All strings: nothing gets silently nulled by a schema
raw = (spark.read.option("header", True).option("inferSchema", False).csv(RAW)
       .withColumn("file", F.col("_metadata.file_name")))
raw.printSchema()
display(raw.limit(10))

total = raw.count()
print(f"Total rows: {total:,}")
display(raw.groupBy("file").count().orderBy("file"))

# COMMAND ----------
# Null/empty rate per column (%)
display(raw.select([
    (F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)) / total * 100).alias(c)
    for c in raw.columns if c != "file"]))

# COMMAND ----------
# Typed view for profiling (casts that fail become null, which we count)
typed = (raw
    .withColumn("ts",  F.to_timestamp("BaseDateTime"))
    .withColumn("lat", F.col("LAT").cast("double"))
    .withColumn("lon", F.col("LON").cast("double"))
    .withColumn("sog", F.col("SOG").cast("double"))
    .withColumn("cog", F.col("COG").cast("double"))
    .withColumn("hdg", F.col("Heading").cast("double")))

def cnt(cond): return F.sum(cond.cast("int"))

# COMMAND ----------
# Main quality profile
display(typed.agg(
    F.countDistinct("MMSI").alias("distinct_vessels"),
    F.min("ts").alias("min_ts"), F.max("ts").alias("max_ts"),
    cnt(F.col("ts").isNull()).alias("unparseable_ts"),
    cnt(F.col("lat").isNull() | F.col("lon").isNull()).alias("null_lat_or_lon"),
    cnt((F.col("lat") < -90) | (F.col("lat") > 90)).alias("bad_lat"),
    cnt((F.col("lon") < -180) | (F.col("lon") > 180)).alias("bad_lon"),
    cnt((F.col("lat") == 91) | (F.col("lon") == 181)).alias("position_sentinel_91_181"),
    cnt(F.col("sog") > 50).alias("speed_over_50"),
    cnt(F.col("sog") == 102.3).alias("sog_sentinel_102_3"),
    cnt(F.col("sog") < 0).alias("speed_negative"),
    cnt(F.col("hdg") == 511).alias("heading_sentinel_511"),
    cnt(F.col("cog") > 360).alias("cog_over_360"),
    cnt(F.length("MMSI") != 9).alias("bad_mmsi_length"),
    cnt(F.col("VesselType") == "0").alias("vessel_type_0"),
))

# COMMAND ----------
# MMSI length distribution
display(typed.groupBy(F.length("MMSI").alias("mmsi_len")).count().orderBy("mmsi_len"))

# COMMAND ----------
# Duplicate (MMSI, timestamp) keys (slow: note the run time)
display(typed.groupBy("MMSI", "ts").count().filter("count > 1")
        .agg(F.count("*").alias("dup_keys"),
             F.sum(F.col("count") - 1).alias("extra_rows")))

# COMMAND ----------
# Rows per event date
display(typed.groupBy(F.to_date("ts").alias("date")).count().orderBy("date"))

# COMMAND ----------
# Check A: are duplicates exact copies? (slow: note the run time)
base = raw.drop("file")
display(base.groupBy(base.columns).count().filter("count > 1")
        .agg(F.count("*").alias("fully_identical_keys")))
# 9,903 = exact copies (safe to drop one). Fewer = conflicting rows, flag those.

# COMMAND ----------
# Check B: what are the fast, non-sentinel rows?
display(typed.filter((F.col("sog") > 50) & (F.col("sog") != 102.3))
        .groupBy("VesselType", F.substring("MMSI", 1, 3).alias("mmsi_prefix"))
        .count().orderBy(F.desc("count")).limit(20))
# MMSIs starting 111 are search-and-rescue aircraft, which legitimately fly fast.

# COMMAND ----------
# Check C: COG edge cases
display(typed.agg(cnt(F.col("cog") == 360).alias("cog_eq_360"),
                  cnt(F.col("cog") > 360).alias("cog_gt_360")))

root
 |-- MMSI: string (nullable = true)
 |-- BaseDateTime: string (nullable = true)
 |-- LAT: string (nullable = true)
 |-- LON: string (nullable = true)
 |-- SOG: string (nullable = true)
 |-- COG: string (nullable = true)
 |-- Heading: string (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Length: string (nullable = true)
 |-- Width: string (nullable = true)
 |-- Draft: string (nullable = true)
 |-- Cargo: string (nullable = true)
 |-- TransceiverClass: string (nullable = true)
 |-- file: string (nullable = false)

Total rows: 49,208,485


In [0]:
# code 02


# Databricks notebook source
import pyspark.sql.functions as F
from delta.tables import DeltaTable

# Paste the NEW (rotated) key into the widget box at the top of the notebook.


storage_account = "vesseldatavd99"

RAW    = f"abfss://raw@{storage_account}.dfs.core.windows.net/AIS_2019_01_*.csv"
BRONZE = f"abfss://bronze@{storage_account}.dfs.core.windows.net/ais_bronze_v2"
BATCH_ID = "batch_2019_01_w1"

# COMMAND ----------
# Read everything as strings (Bronze stays raw) and add metadata columns
df = (spark.read.option("header", True).option("inferSchema", False).csv(RAW)
      .withColumn("source_file", F.col("_metadata.file_name"))
      .withColumn("source_file_date",
                  F.to_date(F.regexp_extract("source_file", r"AIS_(\d{4}_\d{2}_\d{2})", 1), "yyyy_MM_dd"))
      .withColumn("ingestion_id", F.lit(BATCH_ID))
      .withColumn("ingestion_timestamp", F.current_timestamp()))

dates = sorted(r[0].isoformat() for r in df.select("source_file_date").distinct().collect())
print("Dates in this batch:", dates)
assert None not in dates and len(dates) > 0, "Could not parse a date from some file names"
date_list = ",".join(f"'{d}'" for d in dates)

# COMMAND ----------
# Idempotent write: first run creates the table, later runs replace only these dates
import time
t0 = time.time()

writer = df.write.format("delta").partitionBy("source_file_date").mode("overwrite")
if DeltaTable.isDeltaTable(spark, BRONZE):
    writer = writer.option("replaceWhere", f"source_file_date IN ({date_list})")
writer.save(BRONZE)

print(f"Write took {time.time() - t0:.0f} seconds")

# COMMAND ----------
# Verify: rows per date and total (must be 49,208,485 after every run)
bronze = spark.read.format("delta").load(BRONZE)
display(bronze.groupBy("source_file_date").count().orderBy("source_file_date"))
print(f"Total: {bronze.count():,}")

# COMMAND ----------
# File layout: numFiles and sizeInBytes go into docs/BRONZE_RESULTS.md
display(spark.sql(f"DESCRIBE DETAIL delta.`{BRONZE}`"))

# COMMAND ----------
# Pruning test: compare "files read" and "bytes read" in the Spark UI for each query
print(bronze.filter(F.col("source_file_date") == F.to_date(F.lit("2019-01-03"), "yyyy-MM-dd")).count())   # one partition
print(bronze.count())                                                       # all partitions

Dates in this batch: ['2019-01-01', '2019-01-02', '2019-01-03', '2019-01-04', '2019-01-05', '2019-01-06', '2019-01-07']
Write took 475 seconds
Total: 49,208,485
6724018
49208485
